<a href="https://colab.research.google.com/github/21centjoe/1D-synthetic-logic-gates/blob/main/Copy_of_Could_you_tell_it_to_load_from_that_one_website_s_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Here is the updated code formatted specifically for a Google Colab notebook. It includes the code to dynamically load content and campaign references from the provided D&D Beyond forums link, complete with error handling and fallback options so it runs smoothly in your notebook environment.

In [ ]:
# Run this cell first in your Colab notebook to install requirements if needed
!pip install requests beautifulsoup4 -q

import random
import requests
from bs4 import BeautifulSoup

CHARACTERS = ["Unicorn", "Court Jester", "Cat", "Troll", "Goblin", "Knight", "Princess"]
BOARD_SIZE = 252
INITIAL_LIFE_POINTS = 10 # New constant for player's starting life points
STAT_RANGE = (1, 10) # New constant for character stat random range
URL = "https://www.dndbeyond.com/forums/dungeons-dragons-discussion/dungeon-masters-only/43718-list-of-free-dnd-campaigns"

def load_campaign_stories():
    print("Loading campaign stories and credits from D&D Beyond Forums...")
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        response = requests.get(URL, headers=headers, timeout=10)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')
            # Extracting forum links/titles for story hooks
            stories = [a.get_text().strip() for a in soup.find_all('a') if len(a.get_text().strip()) > 15][:10]
            if stories:
                print("Successfully loaded community campaign hooks with credit to D&D Beyond Forums.")
                return stories
    except Exception as e:
        print("Note: Live fetch encountered a network or protection restriction. Using curated community fallback list.")

    return [
        "The Sunless Citadel (Credit: D&D Beyond Forums)",
        "Death House (Credit: D&D Beyond Forums)",
        "A Most Potent Brew (Credit: D&D Beyond Forums)",
        "Wild Sheep Chase (Credit: D&D Beyond Forums)"
    ]

def display_map(players_positions): # Expects {name: position}
    print("\n--- VISUAL MAP ---")
    map_length = 50 # Fixed length for the displayed map segment
    for name, pos in players_positions.items():
        track = ["."] * (BOARD_SIZE + 1)
        clamped_pos = min(pos, BOARD_SIZE)
        track[clamped_pos] = "P"

        start_index = max(0, clamped_pos - map_length // 2)
        end_index = min(BOARD_SIZE + 1, start_index + map_length)

        # Adjust start_index if we hit the end of the board, to ensure map_length is maintained
        if end_index - start_index < map_length:
            start_index = max(0, end_index - map_length)

        display_segment = "".join(track[start_index:end_index])

        # Add visual markers for start and end of the displayed segment
        prefix = "..." if start_index > 0 else ""
        suffix = "..." if end_index <= BOARD_SIZE else ""

        print(f"{name}: {prefix}{display_segment}{suffix} [Pos: {pos}]{' [Finished!]' if pos >= BOARD_SIZE else ''}")
    print("-" * 20)

def present_task(story_hook):
    print(f"\n+--------------------------------------------------+")
    print(f"| STORY HOOK: {story_hook[:36]:<36} |")
    print(f"| TASK BOX: An obstacle blocks your path!          |")
    print(f"| a) Fight your way through with brute force       |")
    print(f"| b) Sneak past quietly using clever tactics       |")
    print(f"| c) Negotiate or use magic to bypass the hazard   |")
    print(f"+--------------------------------------------------+")
    choice = input("Choose your action (a, b, or c): ").lower().strip()
    return choice

def play_game():
    stories = load_campaign_stories()
    print("\nWelcome to Realm Hopper!")

    while True: # Loop for playing multiple games (for reset button functionality)
        # For practice, fix number of players to 2 as requested by removing the input prompt for num_players
        num_players = 2
        print(f"Setting number of players to {num_players} for practice session.")

        players = {} # This will store player objects {name: {'position': X, 'strength': Y, ...}}
        for i in range(num_players):
            print(f"\nAvailable characters: {CHARACTERS}")
            p_name = input(f"Enter name for Player {i+1}: ")
            players[p_name] = {
                'position': 0,
                'strength': random.randint(*STAT_RANGE),
                'dexterity': random.randint(*STAT_RANGE),
                'life_points': INITIAL_LIFE_POINTS
            }
            print(f"{p_name}'s stats: Str:{players[p_name]['strength']}, Dex:{players[p_name]['dexterity']}, HP:{players[p_name]['life_points']}")


        game_active = True
        turn_count = 0 # Initialize turn counter
        while game_active:
            turn_count += 1 # Increment turn counter each round
            print(f"\n--- Turn {turn_count} ---")
            # Need to create a copy of keys to iterate, as players dict might change during iteration (e.g., player death)
            for name in list(players.keys()):
                player_data = players[name]
                print(f"\n{name}'s turn (HP: {player_data['life_points']}, Str: {player_data['strength']}, Dex: {player_data['dexterity']}).")

                action_choice = input("Press Enter to roll the dice, 's' to save progress, or 'q' to quit: ").lower().strip()

                if action_choice == 'q':
                    game_active = False
                    print("Quitting game. Goodbye!")
                    break # Break from player turn loop
                elif action_choice == 's':
                    print(f"Progress saved for {name} (not really, just a placeholder!).")
                    continue # Skip dice roll and task for this turn, let the player choose again or next player turn

                roll = random.randint(1, 6)
                player_data['position'] += roll
                print(f"{name} rolled a {roll}!")

                # Update display_map to use new player data structure
                display_map({n: p_data['position'] for n, p_data in players.items()})

                if player_data['position'] >= BOARD_SIZE:
                    print(f"\nCongratulations! {name} reached the end of the quest and won the game!")
                    game_active = False
                    break # Break from player turn loop

                current_story = random.choice(stories)
                choice = present_task(current_story)

                # Define a simple Difficulty Class (DC) for tasks
                DC = 6 + random.randint(-2, 2) # Random difficulty around 6

                if choice == 'a': # Brute force - Strength check
                    print(f"Attempting Strength check (DC {DC}). Your Strength: {player_data['strength']}")
                    if random.randint(1, 10) + player_data['strength'] >= DC: # Roll d10 + Str
                        print("Brute force succeeds! You gain extra momentum.")
                        player_data['position'] += 2
                    else:
                        print("Brute force fails! You stumble, lose a step, and take 1 damage.")
                        player_data['position'] = max(0, player_data['position'] - 1)
                        player_data['life_points'] -= 1
                elif choice == 'b': # Sneak - Dexterity check
                    print(f"Attempting Dexterity check (DC {DC}). Your Dexterity: {player_data['dexterity']}")
                    if random.randint(1, 10) + player_data['dexterity'] >= DC: # Roll d10 + Dex
                        print("Sneaking successful. You slip past undetected.")
                    else:
                        print("Sneaking fails! You alert enemies and take 1 damage.")
                        player_data['life_points'] -= 1
                elif choice == 'c': # Negotiate/Magic - General check (e.g., d10 + average of stats)
                    print(f"Attempting General skill check (DC {DC}).")
                    # For simplicity, using average of Str and Dex for 'C' check
                    if random.randint(1, 10) + (player_data['strength'] + player_data['dexterity']) // 2 >= DC:
                        print("Magic/Negotiation clears the path smoothly.")
                    else:
                        print("Negotiation fails! You are delayed and take 1 damage.")
                        player_data['life_points'] -= 1
                else:
                    print("Invalid choice, you lose your focus and stay put.")

                if player_data['life_points'] <= 0:
                    print(f"\n{name} has run out of life points and is defeated!")
                    del players[name] # Remove defeated player
                    if not players: # If no players left, game over
                        print("All players defeated. Game over!")
                        game_active = False
                        break # Break from player turn loop

            if not game_active: # If game_active became False in the inner loop, break the outer game loop
                break

        # Ask to play again after game ends or all players are defeated
        if input("\nPlay another game? (yes/no): ").lower().strip() != 'yes':
            print("Thanks for playing Realm Hopper!")
            break # Exit the main game loop that allows multiple games

if __name__ == "__main__":
    play_game()

Loading campaign stories and credits from D&D Beyond Forums...
Successfully loaded community campaign hooks with credit to D&D Beyond Forums.

Welcome to Realm Hopper!
Setting number of players to 2 for practice session.

Available characters: ['Unicorn', 'Court Jester', 'Cat', 'Troll', 'Goblin', 'Knight', 'Princess']
Enter name for Player 1: Tuba
Tuba's stats: Str:4, Dex:4, HP:10

Available characters: ['Unicorn', 'Court Jester', 'Cat', 'Troll', 'Goblin', 'Knight', 'Princess']
Enter name for Player 2: Unicorn 
Unicorn 's stats: Str:7, Dex:10, HP:10

--- Turn 1 ---

Tuba's turn (HP: 10, Str: 4, Dex: 4).
Press Enter to roll the dice, 's' to save progress, or 'q' to quit: Cat
Tuba rolled a 6!

--- VISUAL MAP ---
Tuba: ......P.............................................. [Pos: 6]
Unicorn : P.................................................... [Pos: 0]
--------------------

+--------------------------------------------------+
| STORY HOOK: D&D Beyond Basic Rules               |
| TASK BOX

## Mini-Game: Cavern Crawlers

Welcome to **Cavern Crawlers**! This mini-game brings an 'arcade' feel to your dungeon delves. You are the hero (`^`), positioned at the bottom of the cavern, fending off waves of monstrous crawlers (`M`) as they descend. Fire your arrows (`|`) upwards to destroy them before they reach you.

**Controls:**
- `a`: Move left
- `d`: Move right
- `f`: Fire arrow
- `q`: Quit game

Each turn, the monsters move, and you get to perform one action. Try to survive as long as you can and clear the waves!

In [ ]:
import os
import time
import random

# --- Cavern Crawlers Mini-Game Constants ---
GAME_WIDTH = 30
GAME_HEIGHT = 15
PLAYER_CHAR = '^'
ENEMY_CHAR = 'M'
BULLET_CHAR = '|'
EMPTY_CHAR = ' '

# --- Game State ---
player_pos = GAME_WIDTH // 2
enemies = [] # List of [x, y] coordinates for enemies
bullets = [] # List of [x, y] coordinates for bullets
score = 0
wave = 0
player_lives = 3

def clear_screen():
    # For Colab, print many newlines to 'clear' the previous frame
    print("\n" * 50)

def draw_board():
    board = [[EMPTY_CHAR for _ in range(GAME_WIDTH)] for _ in range(GAME_HEIGHT)]

    # Place player
    board[GAME_HEIGHT - 1][player_pos] = PLAYER_CHAR

    # Place bullets
    for bx, by in bullets:
        if 0 <= by < GAME_HEIGHT and 0 <= bx < GAME_WIDTH:
            board[by][bx] = BULLET_CHAR

    # Place enemies
    for ex, ey in enemies:
        if 0 <= ey < GAME_HEIGHT and 0 <= ex < GAME_WIDTH:
            board[ey][ex] = ENEMY_CHAR

    # Print board
    clear_screen()
    print("╔" + "═" * GAME_WIDTH + "╗")
    for row in board:
        print("║" + "".join(row) + "║")
    print("╚" + "═" * GAME_WIDTH + "╝")
    print(f"Score: {score} | Wave: {wave} | Lives: {player_lives}")

def generate_enemies(num_enemies):
    new_enemies = []
    for _ in range(num_enemies):
        ex = random.randint(0, GAME_WIDTH - 1)
        ey = random.randint(1, GAME_HEIGHT // 3) # Spawn in upper third
        new_enemies.append([ex, ey])
    return new_enemies

def cavern_crawlers_minigame():
    global player_pos, enemies, bullets, score, wave, player_lives

    print("\nStarting Cavern Crawlers! Press 'q' to quit at any time.")
    input("Press Enter to start...")

    player_pos = GAME_WIDTH // 2
    enemies = []
    bullets = []
    score = 0
    wave = 0
    player_lives = 3
    game_over = False

    while not game_over:
        wave += 1
        # More enemies and faster enemies per wave
        enemies.extend(generate_enemies(wave * 2))

        turn_active = True
        while turn_active and not game_over:
            draw_board()

            # Check for player collision with enemies
            for ex, ey in enemies:
                if ey == GAME_HEIGHT - 1 and ex == player_pos:
                    player_lives -= 1
                    print("A crawler reached you!")
                    # Remove colliding enemy to avoid multiple hits for one enemy
                    enemies = [[x, y] for x, y in enemies if not (x == ex and y == ey)]
                    if player_lives <= 0:
                        game_over = True
                        break
            if game_over: break

            # --- Get Player Input ---
            action = input("Action (a:left, d:right, f:fire, q:quit): ").lower().strip()

            if action == 'a':
                player_pos = max(0, player_pos - 1)
            elif action == 'd':
                player_pos = min(GAME_WIDTH - 1, player_pos + 1)
            elif action == 'f':
                bullets.append([player_pos, GAME_HEIGHT - 2]) # Fire bullet just above player
            elif action == 'q':
                game_over = True
                break
            else:
                print("Invalid action. Try again.")
                continue # Don't advance game state on invalid input

            # --- Update Bullets ---
            new_bullets = []
            for bx, by in bullets:
                if by > 0: # Bullet moves up
                    new_bullets.append([bx, by - 1])
            bullets = new_bullets

            # --- Update Enemies ---
            new_enemies = []
            for ex, ey in enemies:
                collided = False
                for bi in range(len(bullets)):
                    bx, by = bullets[bi]
                    if ex == bx and ey == by:
                        score += 10 # Hit!
                        bullets.pop(bi) # Remove bullet
                        collided = True
                        break
                if not collided and ey < GAME_HEIGHT - 1: # Enemy moves down if not hit and not at bottom
                    new_enemies.append([ex, ey + 1])
                elif not collided and ey == GAME_HEIGHT - 1: # Enemy reached bottom
                     # This will be handled by the player collision check above in next iteration.
                     # For now, just let it pass through to be caught by collision.
                     new_enemies.append([ex, ey]) # Keep enemy if it reached the bottom for collision check

            enemies = [e for e in new_enemies if e[1] < GAME_HEIGHT -1 and not (e[0] == player_pos and e[1] == GAME_HEIGHT - 1)] # Remove enemies that passed the player or were hit

            # Remove enemies that reached the bottom boundary if they didn't collide with player
            # This is to prevent an infinite loop if player misses and enemy stays at bottom row.
            enemies = [[x, y] for x, y in enemies if y < GAME_HEIGHT - 1 or (y == GAME_HEIGHT - 1 and x != player_pos)]

            # Check if all enemies in current wave are defeated
            if not enemies and not game_over:
                print(f"Wave {wave} cleared! Preparing for next wave...")
                time.sleep(1) # Pause before next wave
                turn_active = False

    draw_board() # Final draw
    if player_lives <= 0:
        print("\nGAME OVER! You were overwhelmed by the Cavern Crawlers!")
    else:
        print("\nGame ended. Thanks for playing Cavern Crawlers!")
    print(f"Final Score: {score}")

# You can call this function from your main game loop when a mini-game event triggers
# For now, let's call it directly to demonstrate:
# cavern_crawlers_minigame()


## Mini-Game: Whispering Woods: Bridge of Trials

Welcome to the **Whispering Woods: Bridge of Trials**! You find yourself at the entrance of an ancient, rickety bridge spanning a chasm in a deep, dark forest. Choose your champion wisely, for the path ahead is fraught with peril.

Each turn, you'll face a new challenge on the bridge. Your choices, influenced by your chosen character's unique abilities, will determine your success in crossing.

**Character Archetypes:**
-   **Warrior (W):** Strong and resilient, good for physical challenges.
-   **Wizard (Z):** Wields arcane power, adept at magical solutions with the **Elvish Keys of Immortality** and the ability to **Conjure a Dragon** (though its full power might be yet to unleash!). Also grants bonus points!
-   **Dwarf (D):** Tough and knowledgeable of stone, can disrupt foes and endure.
-   **Huntsman (H):** Agile and observant, skilled in tracking and ranged attacks.

Navigate the bridge, overcome encounters, and reach the other side!

In [1]:
import random
import time

# --- Whispering Woods: Bridge of Trials Constants ---
BRIDGE_LENGTH = 10 # Number of segments to cross
INITIAL_HP = 10

ARCHETYPES = {
    'warrior': {'char': 'W', 'abilities': ['Charge', 'Guard'], 'stats': {'str': 8, 'int': 4, 'dex': 5, 'hp': INITIAL_HP}},
    'wizard': {'char': 'Z', 'abilities': ['Elvish Keys', 'Dragon Breath'], 'stats': {'str': 3, 'int': 9, 'dex': 4, 'hp': INITIAL_HP}, 'score_multiplier': 1.5},
    'dwarf': {'char': 'D', 'abilities': ['Stone Skin', 'Minor Tremor'], 'stats': {'str': 7, 'int': 5, 'dex': 4, 'hp': INITIAL_HP}},
    'huntsman': {'char': 'H', 'abilities': ['Mark Target', 'Freezing Shot'], 'stats': {'str': 5, 'int': 4, 'dex': 8, 'hp': INITIAL_HP}}
}

ENCOUNTERS = [
    {
        'name': 'A Gruff Bridge Troll',
        'description': 'A large, moss-covered troll blocks the path, demanding a toll!',
        'options': {
            'warrior': {'action': 'Charge', 'success_msg': 'You bash the troll aside with a mighty charge!', 'fail_msg': 'The troll swats your charge aside, dealing 2 damage.', 'damage': 2, 'move': 2},
            'wizard': {'action': 'Elvish Keys', 'success_msg': 'You channel the ancient Elvish Keys, revealing the troll\'s true nature and charming it to let you pass!', 'fail_msg': 'The Elvish Keys hum softly, but the troll is too stubborn, dealing 1 damage.', 'damage': 1, 'move': 1},
            'dwarf': {'action': 'Minor Tremor', 'success_msg': 'Your tremor shakes the bridge, causing the troll to lose balance. You dash past!', 'fail_msg': 'The troll laughs off your tremor, dealing 2 damage.', 'damage': 2, 'move': 2},
            'huntsman': {'action': 'Freezing Shot', 'success_msg': 'You fire an icy arrow, momentarily freezing the troll in place. A swift bypass!', 'fail_msg': 'Your arrow bounces off the troll, and it retaliates for 1 damage.', 'damage': 1, 'move': 1}
        }
    },
    {
        'name': 'Crumbling Bridge Section',
        'description': 'The wooden planks ahead are rotten and weak, threatening to collapse!',
        'options': {
            'warrior': {'action': 'Guard', 'success_msg': 'You carefully brace each step, navigating the weak planks with precision.', 'fail_msg': 'A plank gives way! You barely cling on, taking 1 damage.', 'damage': 1, 'move': 1},
            'wizard': {'action': 'Dragon Breath', 'success_msg': 'You exhale a burst of arcane energy, solidifying the crumbling planks with raw magic!', 'fail_msg': 'Your magical breath is not enough, and you narrowly avoid falling, taking 1 damage.', 'damage': 1, 'move': 1},
            'dwarf': {'action': 'Stone Skin', 'success_msg': 'Your natural toughness lets you stride confidently across, shrugging off minor collapses.', 'fail_msg': 'Despite your tough hide, a jagged splinter pierces you, dealing 1 damage.', 'damage': 1, 'move': 2},
            'huntsman': {'action': 'Mark Target', 'success_msg': 'You swiftly identify the safest path, marking your steps with expert agility.', 'fail_msg': 'You misjudge a step, slipping and taking 1 damage.', 'damage': 1, 'move': 2}
        }
    },
    {
        'name': 'Gloomfang Bats Swarm',
        'description': 'A cloud of screeching gloomfang bats erupts from below the bridge, flying towards you!',
        'options': {
            'warrior': {'action': 'Guard', 'success_msg': 'You raise your shield, deflecting the furious bat attack!', 'fail_msg': 'The bats overwhelm your guard, scratching you for 2 damage.', 'damage': 2, 'move': 1},
            'wizard': {'action': 'Dragon Breath', 'success_msg': 'You unleash a roaring Dragon Breath, incinerating the bat swarm and clearing the path!', 'fail_msg': 'Your Dragon Breath singes a few bats, but they still manage to bite, dealing 1 damage.', 'damage': 1, 'move': 2},
            'dwarf': {'action': 'Stone Skin', 'success_msg': 'The bats bounce harmlessly off your stone-like skin, unable to pierce it!', 'fail_msg': 'A few persistent bats manage to find weak spots, dealing 1 damage.', 'damage': 1, 'move': 1},
            'huntsman': {'action': 'Freezing Shot', 'success_msg': 'You loose a freezing shot, turning a cluster of bats into icy projectiles that scatter the rest!', 'fail_msg': 'Your shot misses, and the bats nip at you for 1 damage.', 'damage': 1, 'move': 2}
        }
    }
]

def display_bridge(player_pos, player_char):
    bridge = ['-' for _ in range(BRIDGE_LENGTH)]
    if 0 <= player_pos < BRIDGE_LENGTH:
        bridge[player_pos] = player_char
    print("\n--- Bridge of Trials ---")
    print("Start [" + "".join(bridge) + "] End")
    print(f"Position: {player_pos}/{BRIDGE_LENGTH}")

def resolve_encounter(player_archetype, player_full_data, encounter):
    print(f"\nEncounter: {encounter['name']}")
    print(f"\n{encounter['description']}")

    archetype_options = encounter['options'][player_archetype]
    print(f"Your {player_archetype.capitalize()} abilities: (A) {archetype_options['action']}")

    # For simplicity, we'll make a single choice for now, assuming the archetype's primary action
    # In a more complex version, we could offer more choices.
    input(f"Press Enter to use your {archetype_options['action']}...")

    # --- Simple success/fail based on a d10 roll + a relevant stat vs a fixed DC ---
    # This part can be made much more complex and dynamic later
    success_dc = 7 # Base Difficulty Class
    # Example: Warrior might use Strength, Wizard Intelligence, etc. For now, random chance.
    roll = random.randint(1, 10)

    # Dynamically pick a stat for the check based on archetype (simplified)
    if player_archetype == 'warrior': check_stat = player_full_data['stats']['str']
    elif player_archetype == 'wizard': check_stat = player_full_data['stats']['int']
    elif player_archetype == 'dwarf': check_stat = player_full_data['stats']['str'] # Dwarves are strong
    elif player_archetype == 'huntsman': check_stat = player_full_data['stats']['dex']
    else: check_stat = 0 # Fallback

    if roll + check_stat >= success_dc:
        print(f"SUCCESS! {archetype_options['success_msg']}")
        player_full_data['position'] += archetype_options.get('move', 1) # Default move 1 on success
        points_awarded = 10 * player_full_data.get('score_multiplier', 1)
        player_full_data['score'] += points_awarded
        print(f"You gained {int(points_awarded)} points!")
    else:
        print(f"FAILURE! {archetype_options['fail_msg']}")
        player_full_data['current_hp'] -= archetype_options['damage']
        player_full_data['position'] = max(0, player_full_data['position'] - archetype_options.get('move_on_fail', 0)) # Sometimes pushed back

def play_bridge_of_trials():
    global ARCHETYPES # Access global archetypes for selection

    print("\n=== Whispering Woods: Bridge of Trials ===")

    player_archetype = None
    while player_archetype not in ARCHETYPES:
        print("Choose your archetype:")
        for key, value in ARCHETYPES.items():
            print(f" - {key.capitalize()} ({value['char']})" + (" (Bonus Points!) " if 'score_multiplier' in value else ""))
        choice = input("Enter your choice (e.g., 'warrior'): ").lower().strip()
        if choice in ARCHETYPES:
            player_archetype = choice
        else:
            print("Invalid choice. Please pick from the available archetypes.")

    player_data = ARCHETYPES[player_archetype].copy() # Copy archetype data
    player_data['position'] = 0
    player_data['current_hp'] = player_data['stats']['hp']
    player_data['char'] = player_data['char'] # Ensure char is available
    player_data['score'] = 0 # Initialize score
    print(f"You have chosen {player_archetype.capitalize()}! HP: {player_data['current_hp']}, Score: {player_data['score']}")

    game_active = True
    turn = 0

    while game_active:
        turn += 1
        display_bridge(player_data['position'], player_data['char'])
        print(f"\n--- Turn {turn} ({player_archetype.capitalize()} HP: {player_data['current_hp']}, Score: {player_data['score']}) ---")

        if player_data['position'] >= BRIDGE_LENGTH:
            print("\nCONGRATULATIONS! You have successfully crossed the Bridge of Trials!")
            game_active = False
            break

        if player_data['current_hp'] <= 0:
            print("\nGAME OVER! You were defeated on the Bridge of Trials.")
            game_active = False
            break

        #input("Press Enter to continue, or 'q' to quit: ").lower().strip()
        #if input == 'q':
        #    print("Quitting game.")
        #    game_active = False
        #    break
        # Removed the input prompt for turn progression to make it run faster for demonstration
        print("Press Enter to continue...")
        time.sleep(1) # Pause for readability, simulating player action

        # Randomly select an encounter
        current_encounter = random.choice(ENCOUNTERS)

        resolve_encounter(player_archetype, player_data, current_encounter)

        # Position update is now handled within resolve_encounter for successes
        # For failures, position might be pushed back (move_on_fail) or stay same (default 0)
        player_data['position'] = max(0, min(player_data['position'], BRIDGE_LENGTH)) # Keep within bounds

        time.sleep(1) # Pause for readability

    print("\n--- Game End ---")
    print(f"Final Position: {player_data['position']}/{BRIDGE_LENGTH}")
    print(f"Remaining HP: {player_data['current_hp']}")
    print(f"Final Score: {player_data['score']}")

# To play, uncomment the line below:
# play_bridge_of_trials()